# LatentDriver Public Eval (Colab)

This notebook evaluates the **public checkpoints only** under standardized Waymax settings and saves durable artifacts to Drive.

## Evaluation contract

- same validation artifacts
- same `npc_policy_type` per tier
- same seed unless explicitly changed
- same patched upstream fork
- same machine-readable metric schema

Start with `smoke_reactive`, then switch to `full_reactive` or `full_non_reactive` after the smoke path succeeds.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


In [ ]:
INSTALL_RUNTIME = True

if INSTALL_RUNTIME:
    run([sys.executable, "scripts/setup_colab_runtime.py", "--editable-project"], cwd=REPO_DIR)
else:
    print("Skipping runtime installation. Set INSTALL_RUNTIME=True if this is a fresh Colab session.")


In [ ]:
EVAL_TIER = "smoke_reactive"  # switch to full_reactive / full_non_reactive later
EVAL_SEED = 0
MODELS = ["latentdriver_t2_j3", "latentdriver_t2_j4", "plant", "easychauffeur_ppo"]

for model in MODELS:
    print(f"- {model}")
print(json.dumps({"tier": EVAL_TIER, "seed": EVAL_SEED}, indent=2, sort_keys=True))


In [ ]:
import json

preflight = run([
    sys.executable,
    "scripts/run_waymax_eval.py",
    "--model",
    MODELS[0],
    "--tier",
    EVAL_TIER,
    "--seed",
    str(EVAL_SEED),
    "--dry-run",
], cwd=REPO_DIR)
print(preflight)


In [ ]:
import json
import pandas as pd

RUN_EVAL = True
results = []

if RUN_EVAL:
    for index, model in enumerate(MODELS, start=1):
        print(f"\n=== [{index}/{len(MODELS)}] {model} ===")
        output = run([
            sys.executable,
            "scripts/run_waymax_eval.py",
            "--model",
            model,
            "--tier",
            EVAL_TIER,
            "--seed",
            str(EVAL_SEED),
        ], cwd=REPO_DIR)
        payload = json.loads(output)
        results.append(payload)
else:
    print("Skipping evaluation run.")

rows = []
for item in results:
    summary = item.get("summary", {})
    rows.append({
        "model": item["model"],
        "tier": item["tier"],
        "seed": item["seed"],
        "mAR[75:95]": summary.get("mar_75_95"),
        "AR[75:95]": summary.get("ar_75_95"),
        "OR": summary.get("offroad_rate"),
        "CR": summary.get("collision_rate"),
        "PR": summary.get("progress_rate"),
        "run_dir": item["run_dir"],
    })

df = pd.DataFrame(rows)
df


In [ ]:
import pandas as pd

PAPER_REFERENCE = {
    "full_reactive": {
        "plant": {"mAR[75:95]": 75.86, "AR[75:95]": 87.01, "OR": 2.29, "CR": 3.08, "PR": 95.38},
        "easychauffeur_ppo": {"mAR[75:95]": 78.72, "AR[75:95]": 88.66, "OR": 3.95, "CR": 4.72, "PR": 98.26},
        "latentdriver_t2_j4": {"mAR[75:95]": 89.31, "AR[75:95]": 93.79, "OR": 2.59, "CR": 3.22, "PR": 99.50},
        "latentdriver_t2_j3": {"mAR[75:95]": 90.14, "AR[75:95]": 94.31, "OR": 2.22, "CR": 3.13, "PR": 99.64},
    },
    "full_non_reactive": {
        "plant": {"mAR[75:95]": 75.34, "AR[75:95]": 87.39, "OR": 2.15, "CR": 3.80, "PR": 95.11},
        "easychauffeur_ppo": {"mAR[75:95]": 78.33, "AR[75:95]": 88.21, "OR": 3.54, "CR": 4.82, "PR": 97.77},
        "latentdriver_t2_j4": {"mAR[75:95]": 89.63, "AR[75:95]": 94.82, "OR": 2.58, "CR": 2.31, "PR": 99.55},
        "latentdriver_t2_j3": {"mAR[75:95]": 90.38, "AR[75:95]": 95.54, "OR": 2.20, "CR": 2.03, "PR": 99.68},
    },
}

if EVAL_TIER in PAPER_REFERENCE and not df.empty:
    paper_df = pd.DataFrame.from_dict(PAPER_REFERENCE[EVAL_TIER], orient="index").reset_index(names="model")
    compare = df.merge(paper_df, on="model", suffixes=("_observed", "_paper"))
    for metric in ["mAR[75:95]", "AR[75:95]", "OR", "CR", "PR"]:
        compare[f"{metric}_delta"] = compare[f"{metric}_observed"] - compare[f"{metric}_paper"]
    compare
else:
    print("Paper reference comparison is only defined for full_reactive and full_non_reactive.")


## Notes

- `latentdriver_t2_j4` may deviate from the upstream README because this repo standardizes around `batch_dims=[7,125]` for direct comparability, while the upstream ablation for `J=4` uses `batch_dims=[7,150]`.
- Every completed run writes `run_manifest.json`, `metrics.json`, `config_snapshot.json`, `stdout.log`, and `stderr.log` into the Drive-backed results root.